🚀 [Run in JupyterLite](../lite/lab/index.html?path=lecture6.ipynb){target="_blank" .btn .btn-primary}

# Lecture 6: Data Manipulation with Pandas

# SARS-CoV-2 as a research problem

To learn about Pandas we will use SARS-CoV-2 data. Before actually jumping to Pandas let's learn about the coronavirus molecular biology.

The following summary is based on these publications:

- [Masters:2006](http://dx.doi.org/10.1016/S0065-3527(06)66005-3)
- [Fehr and Perlman:2015](http://dx.doi.org/10.1007/978-1-4939-2438-7_1)
- [Sola:2015](https://www.annualreviews.org/doi/full/10.1146/annurev-virology-100114-055218)
- [Kirchdoerfer:2016](http://dx.doi.org/10.1038/nature17200)
- [Walls:2020](http://dx.doi.org/10.1016/j.cell.2020.02.058)
- [Jackson:2022](https://www.nature.com/articles/s41580-021-00418-x)

## Genome organization

All coronaviruses contain non-segmented positive-strand RNA genome approx. 30 kb in length. It is invariably 5'-leader-UTR-replicase-S-E-M-N-3'UTR-poly(A). In addition, it contains a variety of accessory proteins interspersed throughout the genome.

![Genome organization](https://media.springernature.com/original/springer-static/image/chp%3A10.1007%2F978-1-4939-2438-7_1/MediaObjects/317916_1_En_1_Fig1_HTML.gif)

*Genomic organization of representative α, β, and γ CoVs. (From Fehr and Perlman:2015)*

## Virion structure

Coronavirus is a spherical particle approx. 125 nm in diameter. It is covered with S-protein projections giving it an appearance of solar corona - hence the term coronavirus. There are four main structure proteins: spike (S), membrane (M), envelope (E), and nucleocapsid (N).

![Virion structure](https://ars.els-cdn.com/content/image/1-s2.0-S0065352706660053-gr1.jpg)

*Schematic of the coronavirus virion (From Masters:2006)*

---

# Pandas!

> This is an aggregated tutorial relying on material from:
> - [Justin Bois](http://justinbois.github.io/bootcamp/2020/index.html)
> - [BIOS821 course at Duke](https://people.duke.edu/~ccc14/bios-821-2017/index.html)
> - [Pandas documentation](https://pandas.pydata.org/docs/user_guide/index.html/)

Pandas (from "Panel Data") is an essential piece of scientific (and not only) data analysis infrastructure. It is, in essence, a highly optimized library for manipulating very large tables (or "Data Frames").

## Pandas learning resources

- [Getting started](https://pandas.pydata.org/docs/getting_started/index.html#getting-started) - official introduction from Pandas.
- [Data Science Tools](http://people.duke.edu/~ccc14/bios-821-2017/index.html) - Data Science for Biologists from Duke University.
- [Data Carpentry](https://datacarpentry.org/) - a collection of lessons *à la* Software Carpentry.

In [ ]:
# Pandas, conventionally imported as pd
import pandas as pd

Throughout your research career, you will undoubtedly need to handle data, possibly lots of data. The data comes in lots of formats, and you will spend much of your time **wrangling** the data to get it into a usable form.

Pandas is the primary tool in the Python ecosystem for handling data. Its primary object, the `DataFrame` is extremely useful in wrangling data.

# Basics

## The data set

The dataset we will be using is a subset of metadata describing SARS-CoV-2 datasets from the [Sequence Read Archive](https://www.ncbi.nlm.nih.gov/sra).

It is obtained by going to https://www.ncbi.nlm.nih.gov/sra and performing a query with the following search terms: `txid2697049[Organism:noexp]`.

In [ ]:
from urllib.request import urlretrieve
urlretrieve("https://zenodo.org/records/10680001/files/sra_ncov.csv.gz", "sra_ncov.csv.gz")

In [ ]:
import gzip
with gzip.open('sra_ncov.csv.gz', 'rt') as f:
    for i, line in enumerate(f):
        if i >= 3:
            break
        print(line.strip())

## Reading in data

Pandas has a very powerful function, `pd.read_csv()` that can read in a CSV file and store the contents in a convenient data structure called a **data frame**.

In [ ]:
df = pd.read_csv('sra_ncov.csv.gz')

In [ ]:
# View the first few rows
df.head()

## Indexing data frames

The data frame is a convenient data structure for many reasons. Let's start by looking at how data frames are indexed.

**Important**: We index DataFrames by columns, not rows!

In [ ]:
# This gives us a column
df['Run'].head()

In [ ]:
# Access a single value
df['Run'][4]

However, it's better to use `.loc` for accessing data. This gives the location in the data frame we want.

::: {.callout-tip}
## `loc` versus `iloc`

- `loc`: Label-based indexing - use actual row and column labels
- `iloc`: Integer-based indexing - use integer positions
:::

In [ ]:
df.loc[4, 'Run']

In [ ]:
df.iloc[4:6]

In [ ]:
df.iloc[4:6, [0, 2, 4]]

In [ ]:
df.loc[4:6, ['Run', 'size_MB', 'LibraryStrategy']]

## Filtering: Boolean indexing of data frames

Let's say I wanted to pull out accession numbers of runs produced by Pacific Biosciences machines (labeled as `PACBIO_SMRT`). I can use Boolean indexing to specify the row.

In [ ]:
df.loc[df['Platform'] == 'PACBIO_SMRT', 'Run']

In [ ]:
# Pull the whole record
df.loc[df['Platform'] == 'PACBIO_SMRT', :].head(10)

Now, let's pull out all PacBio records that were obtained from Amplicon sequencing. We can use the `&` operator:

In [ ]:
df.loc[(df['Platform'] == 'PACBIO_SMRT') & (df['LibraryStrategy'] == 'AMPLICON'), :].head(3)

In [ ]:
# See how many match
import numpy as np
inds = (df['Platform'] == 'PACBIO_SMRT') & (df['LibraryStrategy'] == 'AMPLICON')
np.unique(inds, return_counts=True)

## Calculating with data frames

Let's add a column that specifies whether or not the corresponding run is above 100 MB:

In [ ]:
# Add the column to the DataFrame
df['Over100Mb'] = df['size_MB'] >= 100

# Take a look
df.head()

## A note about vectorization

Notice how applying the `>=` operator to a `Series` resulted in **elementwise** application. This is called **vectorization**. It means that we do not have to write a `for` loop to do operations on the elements of a `Series`.

Vectorized code is almost always faster because the looping is done with compiled code under the hood.

## Outputting a new CSV file

In [ ]:
df.to_csv('over100Mb_data.csv', index=False)

In [ ]:
with open('over100Mb_data.csv', 'r') as f:
    for i, line in enumerate(f):
        if i >= 3:
            break
        print(line.strip())

---

# Tidy data

[Hadley Wickham](https://en.wikipedia.org/wiki/Hadley_Wickham) wrote a [great article](http://dx.doi.org/10.18637/jss.v059.i10) in favor of "tidy data." Tidy data frames follow the rules:

1. Each variable is a column.
2. Each observation is a row.
3. Each type of observation has its own separate data frame.

A tidy data frame is almost always **much** easier to work with than non-tidy formats.

## Finding unique values and counts

In [ ]:
df = pd.read_csv('https://zenodo.org/records/10680001/files/sra_ncov.csv.gz')
df = df[df['size_MB'] > 0].reset_index(drop=True)

In [ ]:
df['Platform'].unique()

In [ ]:
df['Platform'].value_counts()

## Sorting

In [ ]:
df_subset = df.sample(n=10)
df_subset

In [ ]:
df_subset.sort_index()

In [ ]:
df_subset.sort_values(by=['LibraryLayout', 'size_MB'])

In [ ]:
df_subset.sort_values(by=['LibraryLayout', 'size_MB'], ascending=[True, False])

---

# Split-apply-combine

Let's say we want to compute the total size of SRA runs for each `BioProject`. The strategy is:

1. **Split** the data set up according to the `'BioProject'` field
2. **Apply** a sum function to the split data sets
3. **Combine** the results into a new summary data set

This is the **split-apply-combine** strategy, put forward by Hadley Wickham in [this paper](http://dx.doi.org/10.18637/jss.v040.i01).

## Aggregation

In [ ]:
grouped = df.groupby(['BioProject'])
grouped

In [ ]:
df_sum = pd.DataFrame(grouped['size_MB'].sum())
df_sum.head(10)

In [ ]:
df_sum = df_sum.reset_index()
df_sum.head()

In [ ]:
# Multiple columns in groupby
df.groupby(['BioProject', 'Platform']).sum(numeric_only=True).reset_index().head(10)

In [ ]:
# Descriptive statistics
df.groupby(['BioProject', 'Platform'])['size_MB'].describe().head(10)

In [ ]:
# Custom aggregations
df.groupby(['BioProject', 'Platform']).agg({'size_MB': np.mean, 'Run': 'nunique'}).head(10)

---

## Tidying a data set with melt

The most useful function for tidying data is `pd.melt()`. Let's demonstrate with a coverage dataset:

In [ ]:
df_cov = pd.read_csv('https://zenodo.org/records/10680470/files/coverage.tsv.gz', sep='\t')
df_cov.head()

These data are not tidy. When we melt the data frame, the data within it (called **values**) become a single column. The headers (called **variables**) also become new columns.

![Dataframe melt](https://pandas.pydata.org/docs/_images/07_melt.svg)

In [ ]:
melted = pd.melt(df_cov, 
                 value_name='coverage', 
                 var_name='sample',
                 value_vars=df_cov.columns[3:],
                 id_vars=['start', 'end'])

melted.head()

In [ ]:
melted.groupby(['sample'])['coverage'].describe()

To get back from melted (narrow) format to wide format, use `pivot()`:

![Dataframe pivot](https://pandas.pydata.org/docs/_images/07_pivot.svg)

In [ ]:
melted.pivot(index=['start', 'end'], columns='sample', values='coverage').head()

---

# Working with multiple tables

Working with multiple tables often involves joining them on a common key.

![Left join](https://pandas.pydata.org/docs/_images/08_merge_left.svg)

In [ ]:
df1 = pd.DataFrame({"key": ["A", "B", "C", "D"], "value": np.random.randn(4)})
df2 = pd.DataFrame({"key": ["B", "D", "D", "E"], "value": np.random.randn(4)})

In [ ]:
df1

In [ ]:
df2

## Inner join

![Inner join](https://upload.wikimedia.org/wikipedia/commons/thumb/1/18/SQL_Join_-_07_A_Inner_Join_B.svg/234px-SQL_Join_-_07_A_Inner_Join_B.svg.png)

In [ ]:
pd.merge(df1, df2, on="key")

## Left join

![Left join](https://upload.wikimedia.org/wikipedia/commons/thumb/d/dc/SQL_Join_-_01b_A_Left_Join_B.svg/234px-SQL_Join_-_01b_A_Left_Join_B.svg.png)

In [ ]:
pd.merge(df1, df2, on="key", how="left").fillna('.')

## Right join

![Right join](https://upload.wikimedia.org/wikipedia/commons/thumb/5/5f/SQL_Join_-_03_A_Right_Join_B.svg/234px-SQL_Join_-_03_A_Right_Join_B.svg.png)

In [ ]:
pd.merge(df1, df2, on="key", how="right").fillna('.')

## Full (outer) join

![Full join](https://upload.wikimedia.org/wikipedia/commons/thumb/6/61/SQL_Join_-_05_A_Full_Join_B.svg/234px-SQL_Join_-_05_A_Full_Join_B.svg.png)

In [ ]:
pd.merge(df1, df2, on="key", how="outer").fillna('.')

---

# Putting it all together: Pandas + Altair

## Understanding [Altair](https://altair-viz.github.io/)

Vega-Altair is a declarative statistical visualization library for Python. It offers a powerful and concise grammar that enables you to quickly build a wide range of statistical visualizations.

In [ ]:
import pandas as pd
import altair as alt
from datetime import date
today = date.today()

In [ ]:
# Read a larger dataset
sra = pd.read_csv(
    "https://zenodo.org/records/10680776/files/ena.tsv.gz",
    compression='gzip',
    sep="\t",
    low_memory=False,
    nrows=100000  # Limit rows for faster loading
)

In [ ]:
len(sra)

In [ ]:
sra.sample(5)

## Cleaning the data

In [ ]:
# Convert collection_date to datetime
# Note: errors='coerce' converts unparseable dates to NaT (Not a Time)
# This is acceptable here because we will filter out invalid dates in the next step
sra = sra.assign(collection_date=pd.to_datetime(sra["collection_date"], errors='coerce'))

In [ ]:
print('Earliest entry:', sra['collection_date'].min())
print('Latest entry:', sra['collection_date'].max())

::: {.callout-warning}
## Data Quality

Don't get surprised here - the metadata is only as good as the person who entered it. So, **when you enter metadata for your sequencing data -- pay attention!!!**
:::

In [ ]:
# Filter to valid date range using explicit Timestamp objects for clarity
sra = sra[
    (sra['collection_date'] >= pd.Timestamp('2020-01-01')) 
    & 
    (sra['collection_date'] <= pd.Timestamp('2023-12-31'))
]

In [ ]:
# Aggregate data for heatmap
heatmap_2d = sra.groupby(
    ['instrument_platform', 'library_strategy']
).agg(
    {'run_accession': 'nunique'}
).reset_index()

heatmap_2d

## Plotting the data

In [ ]:
back = alt.Chart(heatmap_2d).mark_rect(opacity=1).encode(
    x=alt.X(
        "instrument_platform:N",
        title="Instrument"
    ),
    y=alt.Y(
        "library_strategy:N",
        title="Strategy",
        axis=alt.Axis(orient='right')
    ),
    color=alt.Color(
        "run_accession:Q",
        title="# Samples",
        scale=alt.Scale(
            scheme="goldred",
            type="log"
        ),
    ),
    tooltip=[
        alt.Tooltip(
            "instrument_platform:N",
            title="Machine"
        ),
        alt.Tooltip(
            "run_accession:Q",
            title="Number of runs"
        ),
        alt.Tooltip(
            "library_strategy:N",
            title="Protocol"
        )
    ]
).properties(
    width=500,
    height=150,
    title={
        "text": ["Breakdown of datasets from ENA",
                 "by Platform and Library Strategy"],
        "subtitle": "(Sample of 100k records)"
    }
)

back

In [ ]:
# Add text labels
front = back.mark_text(
    align="center",
    baseline="middle",
    fontSize=12,
    fontWeight="bold",
).encode(
    text=alt.Text("run_accession:Q", format=",.0f"),
    color=alt.condition(
        alt.datum.run_accession > 200,
        alt.value("white"),
        alt.value("black")
    )
)

# Combine layers
back + front

## Summary

In this lecture, we covered:

1. **Pandas basics**: DataFrames, indexing with `loc` and `iloc`
2. **Boolean indexing**: Filtering data with conditions
3. **Calculations**: Vectorized operations on columns
4. **Tidy data**: Principles of data organization
5. **Split-apply-combine**: Using `groupby()` for aggregations
6. **Reshaping**: `melt()` and `pivot()` for transforming data
7. **Joins**: Combining tables with `merge()`
8. **Visualization**: Creating plots with Altair

These skills form the foundation of data analysis in Python!